# Free flight after gateway preparation

This view replays the cell-13 experiment on the 20-cell Interval Echo federation. The observer prepares one sharp load at a time and releases the first successful preparation. Each exposure uses a fresh copy. The numbers are probabilities calculated by the native model. The table includes preparations that exhaust the retry budget.

GitHub displays the saved output. To rerun it, open this notebook in a checkout of the OPH source with the packages from `model/requirements.txt` available to its Python kernel.

In [1]:
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'model/executable_model/carrier.py').is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from model.candidate_laws.interval_echo.openwave.row_04_compact.gateway_acquisition import prepare_layout, scored_appointments
from model.candidate_laws.interval_echo.openwave.row_04_compact.gateway_bound import SCORE, certify
from model.candidate_laws.interval_echo.openwave.row_04_compact.experiment import gateway_table
from model.executable_model import federation

home = 13
tips = tuple(gateway_table(federation(0))[home])
prepared = prepare_layout(tips)
rows = scored_appointments(prepared, home=home)['readings']
print(f'Home cell {home}, gateway ports {tips}')
print(f'All-time joint home lower bound for this cell: {certify(tips)["joint_home_floor_ppt"]/SCORE:.12f}')
print(f'Source exhaustion: {prepared.exhaustion:.12f}')

Home cell 13, gateway ports (1, 3, 10)
All-time joint home lower bound for this cell: 0.999940596801
Source exhaustion: 0.000059403192


## Free-flight comparison

The coherent and dephased arms start with the same accepted port probabilities. The random-swap control uses the same accepted source and every board seam. The unfiltered control starts at a uniformly sampled sharp port.

In [2]:
arms = {'coherent': 'Coherent', 'dephased': 'Dephased', 'stochastic_swap': 'Random swaps', 'unfiltered_uniform': 'Unfiltered'}
print('Joint home probability')
print('Exposure  ' + ''.join(f'{label:>16}' for label in arms.values()))
for row in rows:
    print(f'{row["t"]:>8}  ' + ''.join(f'{row[arm]["joint_home"]:16.9f}' for arm in arms))

Joint home probability
Exposure          Coherent        Dephased    Random swaps      Unfiltered
       0       0.999940597     0.999940597     0.999940597     1.000000000
       1       0.999940597     0.985616845     0.931493600     0.951215875
       7       0.999940597     0.708517669     0.567254134     0.551620320
      37       0.999940597     0.570143824     0.151118718     0.388503243


## Complete terminal record at exposure 37

Exterior quiet means that preparation succeeded and the load was found outside its home cell. Exhaustion means the source used its 48 attempts without an accepted release. Values are rounded to nine decimal places.

In [3]:
print(f'{"Outcome":<18}' + ''.join(f'{label:>16}' for label in arms.values()))
for index, outcome in enumerate([*(f'Home port {p}' for p in range(12)), 'Exterior quiet', 'Source exhausted']):
    values = []
    for arm in arms:
        read = rows[-1][arm]
        bins = [*read['ports'], read['exterior_quiet'], read['source_exhaustion']]
        values.append(bins[index])
    print(f'{outcome:<18}' + ''.join(f'{v:16.9f}' for v in values))

Outcome                   Coherent        Dephased    Random swaps      Unfiltered
Home port 0            0.000000000     0.009465194     0.012688045     0.014982030
Home port 1            0.000000000     0.010788294     0.012289253     0.017891862
Home port 2            0.000000000     0.013698421     0.012680028     0.019645905
Home port 3            0.000000000     0.006165858     0.012262062     0.015991608
Home port 4            0.201742401     0.099171716     0.012765623     0.049146511
Home port 5            0.184826113     0.090212234     0.012771606     0.047179580
Home port 6            0.184826113     0.090112214     0.012633880     0.046806305
Home port 7            0.201742401     0.099227581     0.012639863     0.047182711
Home port 8            0.113401784     0.063957122     0.012629745     0.046631318
Home port 9            0.000000000     0.013206583     0.012674505     0.019241820
Home port 10           0.000000000     0.009405468     0.012308365     0.016016193
Home